# 01 — Data foundations for Amina's opportunity search

Amina is a fictional new graduate in Nairobi. Before building anything called AI, she needs trustworthy evidence about possible roles. This notebook follows the talk's data topics: data shapes, collection, raw versus trusted storage, quality, cleaning, and visible uncertainty.

Run this notebook in the repository's local Jupyter environment on Windows, macOS, or Linux.

In [ ]:
# Local setup — safe to rerun from the project root or notebooks folder.
import json, os, sys
from pathlib import Path

def find_project(start):
    for folder in [start, *start.parents]:
        if (folder / 'config.yaml').exists() and (folder / 'src').is_dir():
            return folder
    return None

ROOT = find_project(Path.cwd().resolve())
if ROOT is None:
    raise FileNotFoundError('Open Jupyter from the work-opportunity-radar folder, then rerun.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Project: {ROOT}')

## Start with the decision, not the model

Amina's question is: **Which current opportunities should I investigate, what evidence supports them, and what is still unknown?**

That question tells us to retain source links, dates, missing fields, and the original post. A clean-looking table alone is not enough.

In [ ]:
from IPython.display import display
from src import load_config, resolve
from src.profile import load_profile

cfg = load_config()
profile = load_profile(cfg)
display({key: profile[key] for key in ['name', 'location', 'skills', 'target_roles']})

## A proper example: the same evidence in three data shapes

A provider may return **semi-structured** JSON: it has keys, but values can be nested and repeated. We transform selected fields into a **structured** rectangular table with a fixed column for each property. The description inside either representation is still **unstructured** prose.

The example below keeps the facts identical so the difference is about representation, not content.

In [ ]:
import pandas as pd

semi_structured_post = {
    'job': {'id': 'acacia-101', 'title': 'Junior Data Analyst'},
    'organisation': {'name': 'Acacia Analytics'},
    'place': {'city': 'Nairobi', 'country': 'Kenya'},
    'application': {
        'deadline': '2026-09-01',
        'url': 'https://example.org/jobs/acacia-101'
    },
    'skills': ['Python', 'SQL'],
    'description': 'Help the team clean survey data, write SQL queries, and explain findings.'
}
print('SEMI-STRUCTURED JSON (nested keys and a list)')
print(json.dumps(semi_structured_post, indent=2))

In [ ]:
structured_post = pd.DataFrame([{
    'id': semi_structured_post['job']['id'],
    'title': semi_structured_post['job']['title'],
    'company': semi_structured_post['organisation']['name'],
    'location': f"{semi_structured_post['place']['city']}, {semi_structured_post['place']['country']}",
    'skills': ', '.join(semi_structured_post['skills']),
    'deadline': semi_structured_post['application']['deadline'],
    'url': semi_structured_post['application']['url'],
    'description': semi_structured_post['description'],
}])
print('STRUCTURED TABLE (one row, fixed columns)')
display(structured_post)
print('\nUNSTRUCTURED TEXT (meaning is carried by prose)')
print(structured_post.loc[0, 'description'])

## Run the real ingest path

The bundled sample behaves like a provider feed but is fictional, deterministic, and offline. The collector preserves the provider payload before flattening it.

In [ ]:
from src.collect import fetch

cfg['collect']['source'] = 'sample'
cfg['collect']['query'] = ''
jobs = fetch(cfg, write=True, verbose=True)
display(jobs[['title', 'company', 'location', 'deadline', 'source', 'url']].head())

In [ ]:
raw_bundle = json.loads(resolve(cfg['paths']['raw_source_records']).read_text(encoding='utf-8'))
print('RAW PROVIDER RECORD — preserved before cleaning')
print(json.dumps(raw_bundle['records'][0], indent=2, ensure_ascii=False))
print(f"\nRaw evidence: {cfg['paths']['raw_source_records']}")
print(f"Structured collection: {cfg['paths']['collected_jobs']}")

## Measure quality before modelling

Completeness, validity, uniqueness, consistency, and freshness are measurable. A missing value remains evidence about what we do not know; it is not silently dropped.

In [ ]:
from src.quality import print_scorecard, score_dataframe

score = score_dataframe(jobs, cfg)
print_scorecard(score)

In [ ]:
from src.cleaning import clean_collected

trusted = clean_collected(cfg, jobs, verbose=True)
display(trusted[['title', 'location', 'location_clean', 'has_salary',
                 'has_deadline', 'deadline_parsed', 'deadline_unparseable']].head())
print(f"\nTrusted layer: {cfg['paths']['trusted_jobs']}")

## Handoff to the radar

Amina now has traceable data, not an answer. Continue to **02_work_opportunity_radar.ipynb** to separate opportunity classification from candidate-specific relevance, evaluate failure modes, and use generated text with evidence.

**Try it:** Find one missing field and one transformation. Which should be repaired, which should stay unknown, and how could Amina verify it?